In [86]:

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:


import polars as pl
#from polars import selectors as cs
from extract import endpoints, get, get_leagues
from transform import transform_sports, transform_leagues, transform_seasons, transform_divisions
from transform import DivisionSchema, DivisionSeasonsSchema,SportSchema, LeagueSchema, SeasonSchema
from transform import SportsSeasons
from pathlib import Path

In [88]:
data = Path('data')
tables = endpoints.keys()
paths = [data/(table+'.parquet') for table in tables]

In [89]:
for table, path in zip(tables, paths):
    if not path.exists():
        get(table).collect().write_parquet(path)

In [90]:
sports = pl.scan_parquet(data/'sports.parquet')
leagues = pl.scan_parquet(data/'leagues.parquet')
divisions = pl.scan_parquet(data/'divisions.parquet')
seasons = pl.scan_parquet(data/'seasons.parquet')
teams = pl.scan_parquet(data/'teams.parquet')

In [91]:
sports = transform_sports(sports)
sports = SportSchema.validate(sports, cast=True).lazy()

In [92]:
leagues = transform_leagues(leagues)
leagues = LeagueSchema.validate(leagues, cast=True).lazy()

In [93]:
lf = pl.scan_parquet('data/seasons.parquet')
seasons  = transform_seasons(lf)
seasons = SeasonSchema.validate(seasons, cast=True).lazy()

In [94]:
SC, bad = SportsSeasons.filter(
    {
        'sports': sports,
        'seasons': seasons,
    }
)

In [95]:
bad['sports']._df

sport_id,sport_code,sport_name,sport_abbr,sort_order,sport_link,primary_key,sport_id|nullability,sport_code|nullability,sport_code|unique,sport_code|max_length,sport_name|nullability,sport_name|unique,sport_abbr|nullability,sport_abbr|unique,sort_order|nullability,sort_order|unique,sport_link|nullability,sport_link|unique,sport_link|regex,seasons_sports
u32,str,str,str,u32,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
21,"""min""","""Minor League Baseball""","""Minors""",1402,"""/api/v1/sports/21""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
61,"""nlb""","""Negro League Baseball""","""NLB""",2401,"""/api/v1/sports/61""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
32,"""kor""","""Korean Baseball Organization""","""KOR""",2601,"""/api/v1/sports/32""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
31,"""jml""","""Nippon Professional Baseball""","""NPB""",2701,"""/api/v1/sports/31""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
509,"""nae""","""International Baseball (18U)""","""18U""",3503,"""/api/v1/sports/509""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
510,"""nas""","""International Baseball (16 and…","""16U""",3505,"""/api/v1/sports/510""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
6005,"""ame""","""International Baseball (amateu…","""AME""",3509,"""/api/v1/sports/6005""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false
52,"""oly""","""Olympic Baseball""","""OLY""",3511,"""/api/v1/sports/52""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,false


In [99]:
SC.sports.collect()

sport_id,sport_code,sport_name,sport_abbr,sort_order,sport_link
u32,str,str,str,u32,str
1,"""mlb""","""Major League Baseball""","""MLB""",11,"""/api/v1/sports/1"""
11,"""aaa""","""Triple-A""","""AAA""",101,"""/api/v1/sports/11"""
12,"""aax""","""Double-A""","""AA""",201,"""/api/v1/sports/12"""
13,"""afa""","""High-A""","""A+""",301,"""/api/v1/sports/13"""
14,"""afx""","""Single-A""","""A""",401,"""/api/v1/sports/14"""
…,…,…,…,…,…
22,"""bbc""","""College Baseball""","""College""",5101,"""/api/v1/sports/22"""
586,"""hsb""","""High School Baseball""","""H.S.""",6201,"""/api/v1/sports/586"""
576,"""wps""","""Women's Professional Softball""","""WPS""",7001,"""/api/v1/sports/576"""


In [97]:
div, div_seasons = transform_divisions(divisions, leagues)

In [98]:
divisions = DivisionSchema.validate(div, cast=True).lazy()
division_seasons = DivisionSeasonsSchema.validate(div_seasons, cast=True).lazy()